# consip_consumi_convenzione - notebook v0

Validazione della pipeline per fasi: **raw → clean → mart**.

- scopo: verificare che ogni layer sia corretto e coerente con il precedente
- le SQL sono la fonte di verità: i check numerici devono essere letti alla luce di quello che dichiarano
- non sostituisce l'analisi pubblica
- evitare output pesanti o immagini embeddate nel commit

In [1]:
import re
import yaml
import pandas as pd
from pathlib import Path

# --- Unici parametri da impostare manualmente ---
METRICA       = "valore_economico_consumi_totale"  # colonna numerica principale nel mart
METRICA_CLEAN = "valore_economico_consumi"  # colonna corrispondente nel clean

# --- Anchor: usa il path del notebook se disponibile (VSCode), altrimenti cwd ---
try:
    _start = Path(__vsc_ipynb_file__).resolve().parent  # VSCode Jupyter
except NameError:
    _start = Path.cwd().resolve()

# Walk up finché non troviamo dataset.yml
candidate_dir = None
for probe in [_start, *_start.parents]:
    if (probe / "dataset.yml").exists():
        candidate_dir = probe
        break

if candidate_dir is None:
    raise FileNotFoundError(f"dataset.yml non trovato risalendo da {_start}")

cfg = yaml.safe_load((candidate_dir / "dataset.yml").read_text())

SLUG       = cfg["dataset"]["name"]
ANNO_RUN   = cfg["dataset"]["years"][-1]
MART_TABLE = cfg["mart"]["tables"][0]["name"]
ENCODING   = cfg.get("clean", {}).get("read", {}).get("encoding", "utf-8")
DELIM      = cfg.get("clean", {}).get("read", {}).get("delim", ",")
HEADER     = cfg.get("clean", {}).get("read", {}).get("header", True)
SKIP       = cfg.get("clean", {}).get("read", {}).get("skip", 0)

DI_ROOT   = (candidate_dir / cfg["root"]).resolve()
RAW_DIR   = DI_ROOT / "data" / "raw"   / SLUG / str(ANNO_RUN)
CLEAN_DIR = DI_ROOT / "data" / "clean" / SLUG / str(ANNO_RUN)
MART_DIR  = DI_ROOT / "data" / "mart"  / SLUG / str(ANNO_RUN)
SQL_DIR   = candidate_dir / "sql"

print(f"slug      : {SLUG}")
print(f"anno_run  : {ANNO_RUN}")
print(f"mart table: {MART_TABLE}")
print(f"encoding  : {ENCODING}  delim: {repr(DELIM)}")
print(f"header    : {HEADER}  skip: {SKIP}")

slug      : consip_consumi_convenzione
anno_run  : 2025
mart table: mart_consumi_convenzione
encoding  : utf-8  delim: ','
header    : True  skip: 0


## SQL di riferimento

Le SQL sono la fonte di verità per capire cosa deve contenere ogni layer.
Leggile prima di interpretare i check numerici.

In [2]:
for sql_file in sorted(SQL_DIR.glob("*.sql")):
    print(f"{'='*60}")
    print(f"  {sql_file.name}")
    print(f"{'='*60}")
    print(sql_file.read_text())
    print()

  clean.sql
select
  {year}::INTEGER as anno_riferimento,
  trim("Tipologia_Amministrazione") as tipologia_amministrazione,
  trim("Regione_PA") as regione_pa,
  trim("Provincia_PA") as provincia_pa,
  trim("Sigla_provincia_PA") as sigla_provincia_pa,
  trim("Regione_Fornitore") as regione_fornitore,
  trim("Convenzione") as convenzione,
  trim("Lotto") as lotto,
  try_cast(replace(trim("Valore_economico_consumi"), ',', '.') as double) as valore_economico_consumi,
  "Numero_Ordini_con_consumi"::BIGINT as numero_ordini_con_consumi,
  "N_PA_con_consumi"::BIGINT as n_pa_con_consumi,
  "N_PO_con_consumi"::BIGINT as n_po_con_consumi
from raw_input


  mart.sql
select
  anno_riferimento,
  regione_pa,
  provincia_pa,
  sigla_provincia_pa,
  tipologia_amministrazione,
  regione_fornitore,
  sum(valore_economico_consumi) as valore_economico_consumi_totale,
  sum(numero_ordini_con_consumi) as numero_ordini_totale,
  count(distinct convenzione) as numero_convenzioni_distinte,
  count(distinct lo

## 1. Raw

Verifica che il file raw esista, sia leggibile con i parametri dichiarati in `dataset.yml` e abbia il numero di righe atteso.

In [3]:
raw_files = sorted(RAW_DIR.glob("*.csv")) + sorted(RAW_DIR.glob("*.xlsx")) + sorted(RAW_DIR.glob("*.parquet"))
if not raw_files:
    raise FileNotFoundError(f"Nessun file raw trovato in {RAW_DIR}")

raw_path = raw_files[0]
print(f"File: {raw_path.name}  ({raw_path.stat().st_size / 1024:.0f} KB)")

N_RAW = None
raw_full = None
try:
    if raw_path.suffix == ".parquet":
        raw_full = pd.read_parquet(raw_path)
    elif raw_path.suffix in (".csv", ".tsv"):
        header_row = 0 if HEADER else None
        raw_full = pd.read_csv(raw_path, encoding=ENCODING, sep=DELIM, header=header_row, skiprows=SKIP)
    elif raw_path.suffix == ".xlsx":
        raw_full = pd.read_excel(raw_path, header=0 if HEADER else None, skiprows=SKIP)
    N_RAW = len(raw_full)
    print(f"Righe raw   : {N_RAW}")
    print(f"Colonne raw : {len(raw_full.columns)} -> {list(raw_full.columns)}")
    print("Raw caricato OK")
    raw_full.head(3)
except Exception as e:
    print(f"WARNING: raw non leggibile con pandas ({e})")
    print("-> Skip raw-load, il clean parquet e gia validato dalla pipeline.")
    N_RAW = None


File: consumi-in-convenzione-2025.csv  (1071 KB)
Righe raw   : 7115
Colonne raw : 12 -> ['#Anno_Riferimento', 'Tipologia_Amministrazione', 'Regione_PA', 'Provincia_PA', 'Sigla_provincia_PA', 'Regione_Fornitore', 'Convenzione', 'Lotto', 'Valore_economico_consumi', 'Numero_Ordini_con_consumi', 'N_PA_con_consumi', 'N_PO_con_consumi']
Raw caricato OK


## 2. Clean

Verifica schema e null. I null su colonne `try_cast` sono attesi se il raw contiene valori non parsabili.
Il confronto righe raw vs clean mostra quante righe sono state filtrate dal `WHERE` della clean.sql.

In [4]:
clean_files = sorted(CLEAN_DIR.glob("*.parquet"))
if not clean_files:
    raise FileNotFoundError(f"Clean non trovato in {CLEAN_DIR}. Eseguire: toolkit run clean")

clean = pd.read_parquet(clean_files[0])
N_CLEAN = len(clean)

print(f"Righe clean : {N_CLEAN}")
print(f"Colonne     : {list(clean.columns)}")
clean.head(3)

Righe clean : 7115
Colonne     : ['anno_riferimento', 'tipologia_amministrazione', 'regione_pa', 'provincia_pa', 'sigla_provincia_pa', 'regione_fornitore', 'convenzione', 'lotto', 'valore_economico_consumi', 'numero_ordini_con_consumi', 'n_pa_con_consumi', 'n_po_con_consumi']


,anno_riferimento,tipologia_amministrazione,regione_pa,provincia_pa,sigla_provincia_pa,regione_fornitore,convenzione,lotto,valore_economico_consumi,numero_ordini_con_consumi,n_pa_con_consumi,n_po_con_consumi
0,2025,ISTITUTI/ENTI DI RICERCA,SARDEGNA,OGLIASTRA,OG,SARDEGNA,CARBURANTI EXTRARETE 13 E GASOLIO DA RISCALDAM...,GASOLIO DA RISCALDAMENTO SARDEGNA,3841.24,1,1,1
1,2025,ISTITUTI/ENTI DI RICERCA,ABRUZZO,L'AQUILA,AQ,LOMBARDIA,GAS NATURALE 15,"ABRUZZO, MOLISE",133340.40,1,1,1
2,2025,ISTITUTI/ENTI DI RICERCA,CAMPANIA,NAPOLI,NA,LOMBARDIA,ENERGIA ELETTRICA 22,CAMPANIA,221029.95,4,3,3


In [5]:
if N_RAW is not None:
    dropped = N_RAW - N_CLEAN
    dropped_pct = dropped / N_RAW * 100 if N_RAW > 0 else 0
    print(f"Righe raw         : {N_RAW}")
    print(f"Righe clean       : {N_CLEAN}")
    print(f"Righe filtrate    : {dropped} ({dropped_pct:.1f}%)")
    print()
    if dropped > 0:
        print("-> Verificare la WHERE in clean.sql per capire quali righe vengono escluse.")
    else:
        print("-> Nessuna riga filtrata: clean e fedele al raw.")
else:
    print(f"Raw non caricato (parsing error) -- righe clean: {N_CLEAN}")
    print("-> Check raw-vs-clean saltato. Clean gia validato dalla pipeline.")

print("\nNull per colonna clean:")
null_counts = clean.isnull().sum()
print(null_counts[null_counts > 0].to_string() if null_counts.any() else "  nessuno")


Righe raw         : 7115
Righe clean       : 7115
Righe filtrate    : 0 (0.0%)

-> Nessuna riga filtrata: clean e fedele al raw.

Null per colonna clean:
  nessuno


## 3. Mart

Verifica unicità sulle chiavi del GROUP BY, anni presenti, null e consistenza dei totali con il clean.

In [6]:
mart_path = MART_DIR / f"{MART_TABLE}.parquet"
if not mart_path.exists():
    raise FileNotFoundError(f"Mart non trovato: {mart_path}. Eseguire: toolkit run mart")

mart = pd.read_parquet(mart_path)
print(f"Righe mart  : {len(mart)}")
print(f"Colonne     : {list(mart.columns)}")
mart.head(3)

Righe mart  : 3778
Colonne     : ['anno_riferimento', 'regione_pa', 'provincia_pa', 'sigla_provincia_pa', 'tipologia_amministrazione', 'regione_fornitore', 'valore_economico_consumi_totale', 'numero_ordini_totale', 'numero_convenzioni_distinte', 'numero_lotti_distinti', 'numero_pa_totale', 'numero_po_totale']


,anno_riferimento,regione_pa,provincia_pa,sigla_provincia_pa,tipologia_amministrazione,regione_fornitore,valore_economico_consumi_totale,numero_ordini_totale,numero_convenzioni_distinte,numero_lotti_distinti,numero_pa_totale,numero_po_totale
0,2025,ABRUZZO,CHIETI,CH,AZIENDE ED ENTI TERRITORIALI DI SERVIZI PUBBLICI,LOMBARDIA,6405131.84,32.0,4,2,9.0,9.0
1,2025,ABRUZZO,CHIETI,CH,AZIENDE ED ENTI TERRITORIALI DI SERVIZI PUBBLICI,VENETO,1466697.41,6.0,3,2,4.0,4.0
2,2025,ABRUZZO,CHIETI,CH,AZIENDE ED ENTI TERRITORIALI DI SERVIZI PUBBLICI,PIEMONTE,25366.40,4.0,1,1,4.0,4.0


In [7]:
# Estrai chiavi GROUP BY dalla mart.sql per il check di unicità.
# Per query con CTE, cerca GROUP BY solo nella SELECT finale (dopo tutti i CTE).
# Se la SELECT finale non ha GROUP BY, il check di unicità va fatto manualmente.
mart_sql_path = SQL_DIR / Path(cfg["mart"]["tables"][0]["sql"]).name
groupby_keys = []
if mart_sql_path.exists():
    sql_text = mart_sql_path.read_text()

    # Individua dove inizia la SELECT finale (depth 0, dopo eventuali CTE)
    sql_body = sql_text
    if re.search(r'\bwith\b', sql_body, re.IGNORECASE):
        depth = 0
        final_select_pos = None
        for tok in re.finditer(r'[()]|\bselect\b', sql_body, re.IGNORECASE):
            ch = tok.group()
            if ch == '(':
                depth += 1
            elif ch == ')':
                depth -= 1
            elif ch.lower() == 'select' and depth == 0:
                final_select_pos = tok.start()
        if final_select_pos is not None:
            sql_body = sql_body[final_select_pos:]

    match = re.search(r'group\s+by\s+([\d,]+)', sql_body, re.IGNORECASE)
    if match:
        positions = [int(p.strip()) for p in match.group(1).split(',')]
        groupby_keys = [mart.columns[i - 1] for i in positions if i <= len(mart.columns)]
    else:
        match = re.search(r'group\s+by\s+((?:[\w.]+(?:\s*,\s*)?)+)', sql_body, re.IGNORECASE)
        if match:
            groupby_keys = [k.strip().split('.')[-1] for k in match.group(1).split(',')]
            groupby_keys = [k for k in groupby_keys if k in mart.columns]

if groupby_keys:
    dups = mart.duplicated(subset=groupby_keys).sum()
    print(f"Chiavi GROUP BY (SELECT finale): {groupby_keys}")
    print(f"Duplicati                       : {dups}")
    assert dups == 0, f"FAIL: {dups} righe duplicate sulle chiavi del mart -- verificare mart.sql"
    print("OK: nessun duplicato sulle chiavi.")
else:
    print("GROUP BY non trovato nella SELECT finale -- verificare manualmente unicita'.")

Chiavi GROUP BY (SELECT finale): ['anno_riferimento', 'regione_pa', 'provincia_pa', 'sigla_provincia_pa', 'tipologia_amministrazione', 'regione_fornitore']
Duplicati                       : 0
OK: nessun duplicato sulle chiavi.


In [8]:
if "anno" in mart.columns:
    anni_mart = sorted(mart["anno"].unique())
    print(f"Anni nel mart ({len(anni_mart)}): {anni_mart}")

null_mart = mart.isnull().sum()
print("\nNull per colonna mart:")
print(null_mart[null_mart > 0].to_string() if null_mart.any() else "  nessuno")


Null per colonna mart:
  nessuno


In [9]:
if METRICA in mart.columns and METRICA_CLEAN in clean.columns:
    tot_mart  = mart[METRICA].sum()
    tot_clean = clean[METRICA_CLEAN].sum()
    delta_pct = abs(tot_mart - tot_clean) / tot_clean * 100 if tot_clean != 0 else float("nan")
    print(f"Totale mart  ({METRICA})      : {tot_mart:,.2f}")
    print(f"Totale clean ({METRICA_CLEAN}): {tot_clean:,.2f}")
    print(f"Delta %: {delta_pct:.4f}%")
    assert delta_pct < 0.01, f"FAIL: delta {delta_pct:.2f}% -- verificare la pipeline"
    print("OK: i totali coincidono.")
else:
    print(f"Colonne non trovate -- aggiornare METRICA ('{METRICA}') e METRICA_CLEAN ('{METRICA_CLEAN}').")


Totale mart  (valore_economico_consumi_totale)      : 3,062,281,077.03
Totale clean (valore_economico_consumi): 3,062,281,077.03
Delta %: 0.0000%
OK: i totali coincidono.


In [10]:
PERIMETRO = "Consumi delle pubbliche amministrazioni per convenzioni CONSiP — valore economico aggregato per anno, categoria merceologica e amministrazione"

print(f"Slug            : {SLUG}")
print(f"Anno run        : {ANNO_RUN}")
print(f"Tabella mart    : {MART_TABLE}")
print(f"Metrica mart    : {METRICA}")
print(f"Metrica clean   : {METRICA_CLEAN}")
print(f"Perimetro       : {PERIMETRO}")

Slug            : consip_consumi_convenzione
Anno run        : 2025
Tabella mart    : mart_consumi_convenzione
Metrica mart    : valore_economico_consumi_totale
Metrica clean   : valore_economico_consumi
Perimetro       : Consumi delle pubbliche amministrazioni per convenzioni CONSiP — valore economico aggregato per anno, categoria merceologica e amministrazione
